# Public CMU FootNet training
Select **Runtime → Change runtime type → T4 GPU**, then Run all.
This notebook downloads real CMU/COCO data, trains a six-slot pose model, evaluates
on separate images and exports a candidate ONNX. It does not train metric sizing.
CMU annotations: CC BY 4.0. Retain original COCO image licenses and attribution.
K2/K3 are unlabelled; no guessed metatarsal coordinates are used.
The annotation mirror is pinned; its license does not supersede upstream terms.


In [ ]:
%pip install -q ultralytics==8.4.143 onnx onnxruntime pillow pyyaml


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before the full run.')
print(torch.cuda.get_device_name(0))


In [ ]:
%%writefile prepare_public_cmu.py
"""Download real CMU/COCO foot annotations and build a reproducible YOLO pose subset.

No synthetic landmarks are created. K2/K3 (metatarsal heads) are unlabelled.
COCO contains no reliable subject IDs or metric foot measurements.
"""
import argparse
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import json
import math
from pathlib import Path
import random
import time
import urllib.request

MIRROR_REVISION = "5d99ff365a82073704e6feda1a4412065a0b62b1"
MIRROR = f"https://raw.githubusercontent.com/Eva20150932/coco-foot-and-leg/{MIRROR_REVISION}"
SOURCE = "https://cmu-perceptual-computing-lab.github.io/foot_keypoint_dataset/"
KEYPOINTS = ["heel", "ankle", "medial_ball_unlabelled", "lateral_ball_unlabelled", "big_toe", "small_toe"]


def download(url, destination):
    if destination.exists() and destination.stat().st_size:
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    last_error = None
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "FootNet-public-research/1.0"})
            with urllib.request.urlopen(request, timeout=30) as response, temporary.open("wb") as output:
                while chunk := response.read(1024 * 1024):
                    output.write(chunk)
            temporary.replace(destination)
            return
        except Exception as error:
            last_error = error
            time.sleep(attempt + 1)
    raise RuntimeError(f"Download failed: {url}: {last_error}")


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def foot_points(annotation, side):
    raw = annotation.get("keypoints", [])
    if len(raw) not in (18, 69):
        return None
    points = [raw[i:i + 3] for i in range(0, len(raw), 3)]
    offset = 17 if len(points) == 23 else 0
    big, small, heel = points[offset + side * 3:offset + side * 3 + 3]
    ankle = points[15 + side] if offset else [0, 0, 0]
    if not all(p[2] > 0 and all(math.isfinite(v) for v in p) for p in [heel, big, small]):
        return None
    if math.hypot(big[0] - heel[0], big[1] - heel[1]) < 20:
        return None
    return [heel, ankle, [0, 0, 0], [0, 0, 0], big, small]


def bbox_for_points(points, width, height):
    visible = [p for p in points if p[2] > 0]
    xs, ys = [p[0] for p in visible], [p[1] for p in visible]
    length = math.hypot(points[4][0] - points[0][0], points[4][1] - points[0][1])
    padding = max(6, length * 0.22)
    return [max(0, min(xs) - padding), max(0, min(ys) - padding),
            min(width, max(xs) + padding), min(height, max(ys) + padding)]


def convert_image(image_info, annotations, raw_path, output, split, coco_split):
    from PIL import Image
    with Image.open(raw_path) as image:
        image = image.convert("RGB")
        width, height = image.size
        if (width, height) != (image_info["width"], image_info["height"]):
            raise ValueError("Image dimensions disagree with annotations")
        feet = []
        for annotation in annotations:
            if annotation.get("iscrowd", 0):
                continue
            for side in (0, 1):
                points = foot_points(annotation, side)
                if points and all(0 <= p[0] <= width and 0 <= p[1] <= height for p in points if p[2] > 0):
                    feet.append((points, annotation["id"], "left" if side == 0 else "right"))
        if not feet:
            return None
        boxes = [bbox_for_points(points, width, height) for points, _, _ in feet]
        # One crop per source image keeps every annotated foot in that crop.
        x0 = max(0, math.floor(min(b[0] for b in boxes) - 24))
        y0 = max(0, math.floor(min(b[1] for b in boxes) - 24))
        x1 = min(width, math.ceil(max(b[2] for b in boxes) + 24))
        y1 = min(height, math.ceil(max(b[3] for b in boxes) + 24))
        crop_width, crop_height = x1 - x0, y1 - y0
        labels = []
        for (points, _, _), box in zip(feet, boxes):
            bx0, by0, bx1, by1 = box
            values = [0, ((bx0 + bx1) / 2 - x0) / crop_width, ((by0 + by1) / 2 - y0) / crop_height,
                      (bx1 - bx0) / crop_width, (by1 - by0) / crop_height]
            for px, py, visibility in points:
                values.extend([(px - x0) / crop_width, (py - y0) / crop_height, visibility] if visibility else [0, 0, 0])
            labels.append(" ".join(str(v) if i == 0 else f"{v:.6f}" for i, v in enumerate(values)))
        stem = f"cmu_{coco_split}_{image_info['id']:012d}"
        target = output / "images" / split / f"{stem}.jpg"
        image.crop((x0, y0, x1, y1)).save(target, quality=95)
        (output / "labels" / split / f"{stem}.txt").write_text("\n".join(labels) + "\n", encoding="utf-8")
    return {"image_id": image_info["id"], "coco_split": coco_split, "split": split,
            "file": f"images/{split}/{stem}.jpg", "image_url": image_info.get("coco_url"),
            "image_license_id": image_info.get("license"), "original_sha256": sha256(raw_path),
            "crop": [x0, y0, x1, y1], "foot_count": len(feet),
            "annotation_ids": sorted({aid for _, aid, _ in feet})}


def prepare(output, train_images=600, val_images=80, test_images=80, seed=42, workers=8):
    output = Path(output).resolve()
    output.mkdir(parents=True, exist_ok=True)
    for split in ("train", "val", "test"):
        for folder in ("images", "labels"):
            destination = output / folder / split
            destination.mkdir(parents=True, exist_ok=True)
            if list(destination.glob("*.jpg")) or list(destination.glob("*.txt")):
                raise RuntimeError(f"Use a new output directory to prevent stale sample mixing: {destination}")
    if min(train_images, val_images, test_images) < 1:
        raise ValueError("Every split needs at least one image")
    rng = random.Random(seed)
    manifests, errors, annotation_sources, image_licenses = [], [], [], {}
    for coco_split in ("train2017", "val2017"):
        name = f"person_keypoints_{coco_split}_foot_v1.json"
        annotation_path = output / "raw" / name
        url = f"{MIRROR}/{name}"
        print(f"Downloading {coco_split} annotations from pinned CMU mirror", flush=True)
        download(url, annotation_path)
        document = json.loads(annotation_path.read_text(encoding="utf-8"))
        annotation_sources.append({"url": url, "sha256": sha256(annotation_path)})
        image_licenses[coco_split] = document.get("licenses", [])
        grouped = defaultdict(list)
        for annotation in document["annotations"]:
            grouped[annotation["image_id"]].append(annotation)
        candidates = [image for image in document["images"] if any(
            foot_points(annotation, side) for annotation in grouped[image["id"]] for side in (0, 1))]
        rng.shuffle(candidates)
        limits = {"train": train_images} if coco_split == "train2017" else {"val": val_images, "test": test_images}
        required = sum(limits.values())
        candidates = candidates[:required]
        jobs = []
        cursor = 0
        for split, count in limits.items():
            for image in candidates[cursor:cursor + count]:
                jobs.append((split, image))
            cursor += count

        def work(job):
            split, image = job
            raw_path = output / "raw" / coco_split / f"{image['id']:012d}.jpg"
            # COCO's official download endpoint uses HTTP and has no account requirement.
            image_url = f"http://images.cocodataset.org/{coco_split}/{image['id']:012d}.jpg"
            download(image_url, raw_path)
            record = convert_image(image, grouped[image["id"]], raw_path, output, split, coco_split)
            if record:
                record["download_url"] = image_url
            return record

        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(work, job): job for job in jobs}
            for index, future in enumerate(as_completed(futures), 1):
                try:
                    record = future.result()
                    if record:
                        manifests.append(record)
                except Exception as error:
                    errors.append({"image_id": futures[future][1]["id"], "error": str(error)})
                if index % 50 == 0 or index == len(futures):
                    print(f"{coco_split}: {index}/{len(futures)} processed, total {len(manifests)} images", flush=True)
    counts = {split: sum(r["split"] == split for r in manifests) for split in ("train", "val", "test")}
    foot_counts = {split: sum(r["foot_count"] for r in manifests if r["split"] == split) for split in counts}
    report = {"source": SOURCE, "annotation_license": "CC BY 4.0 (CMU); mirror does not replace upstream terms",
              "mirror_revision": MIRROR_REVISION, "annotation_sources": annotation_sources,
              "image_licenses": image_licenses, "seed": seed, "image_counts": counts, "foot_counts": foot_counts,
              "keypoints": KEYPOINTS, "unlabelled_keypoints": [2, 3], "metric_ground_truth": False,
              "split_policy": "Official COCO train vs val; val image IDs partitioned into validation/test. No subject identity guarantee.",
              "records": sorted(manifests, key=lambda r: (r["split"], r["image_id"])), "download_errors": errors}
    (output / "manifest.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    if any(count < 1 for count in counts.values()):
        raise RuntimeError(f"Empty split: {counts}. Inspect manifest download_errors.")
    yaml = f"path: {output.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\nkpt_shape: [6, 3]\nflip_idx: [0, 1, 2, 3, 4, 5]\nnames:\n  0: foot\n"
    (output / "data.yaml").write_text(yaml, encoding="utf-8")
    print(json.dumps({"images": counts, "feet": foot_counts, "errors": len(errors), "data": str(output / "data.yaml")}), flush=True)
    return report


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--output", default="training/public_cmu_pilot")
    parser.add_argument("--train-images", type=int, default=600)
    parser.add_argument("--val-images", type=int, default=80)
    parser.add_argument("--test-images", type=int, default=80)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--workers", type=int, default=8)
    args = parser.parse_args()
    prepare(args.output, args.train_images, args.val_images, args.test_images, args.seed, args.workers)


In [ ]:
%%writefile train_public_footnet.py
"""Train/evaluate a public-data pilot without replacing the app's existing model."""
import argparse
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import time


def audit_dataset(data_path):
    import yaml
    config = yaml.safe_load(data_path.read_text(encoding="utf-8"))
    root = Path(config["path"]).resolve()
    manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
    if manifest.get("metric_ground_truth") is not False:
        raise ValueError("This pipeline expects nonmetric public pose data")
    if config.get("kpt_shape") != [6, 3] or config.get("names") != {0: "foot"}:
        raise ValueError("Expected the web engine's single-class, six-slot pose layout")
    seen = set()
    hash_splits = {}
    actual_counts = {split: 0 for split in ("train", "val", "test")}
    actual_feet = dict(actual_counts)
    for record in manifest["records"]:
        if record["split"] not in actual_counts:
            raise ValueError("Invalid dataset split")
        identity = record.get("source_identity", (record.get("coco_split"), record["image_id"]))
        if isinstance(identity, list):
            identity = tuple(identity)
        if identity in seen:
            raise ValueError("Source image duplicated across dataset records")
        seen.add(identity)
        image_hash = record.get('original_sha256')
        if image_hash and image_hash in hash_splits and hash_splits[image_hash] != record['split']:
            raise ValueError('Identical image contents appear in different splits')
        if image_hash:
            hash_splits[image_hash] = record['split']
        if "coco_split" in record and record["split"] == "train" and record["coco_split"] != "train2017":
            raise ValueError("Training must use official train images only")
        image = (root / record["file"]).resolve()
        if not image.is_relative_to(root):
            raise ValueError("Dataset image path escapes dataset directory")
        label = root / "labels" / record["split"] / (image.stem + ".txt")
        if not image.exists() or not label.exists():
            raise FileNotFoundError(image)
        if record.get("prepared_sha256") and hashlib.sha256(image.read_bytes()).hexdigest() != record["prepared_sha256"]:
            raise ValueError("Prepared dataset image hash has changed")
        actual_counts[record["split"]] += 1
        foot_count = 0
        for line in label.read_text().splitlines():
            values = [float(v) for v in line.split()]
            if (len(values) != 23 or not all(math.isfinite(v) for v in values)
                    or values[0] != 0 or any(not 0 <= v <= 1 for v in values[1:5])
                    or values[3] <= 0 or values[4] <= 0):
                raise ValueError(f"Invalid YOLO label: {label}")
            for index in range(6):
                x, y, visibility = values[5 + index * 3:8 + index * 3]
                if visibility not in (0, 1, 2) or not 0 <= x <= 1 or not 0 <= y <= 1:
                    raise ValueError(f"Invalid keypoint: {label}")
                if index in (2, 3) and visibility != 0:
                    raise ValueError("Unobserved metatarsals must not be fabricated")
            foot_count += 1
        if foot_count != record["foot_count"]:
            raise ValueError("Manifest foot count does not match labels")
        actual_feet[record["split"]] += foot_count
    if actual_counts != manifest["image_counts"] or actual_feet != manifest["foot_counts"]:
        raise ValueError("Manifest counts do not match actual dataset files")
    if any(count < 1 for count in actual_counts.values()):
        raise ValueError("All three splits are required")
    return manifest


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True)
    parser.add_argument("--weights", default="training/yolov8n-pose.pt")
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--imgsz", type=int, default=256)
    parser.add_argument("--device", default="cpu")
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--freeze", type=int, default=10)
    parser.add_argument("--project", default="training/runs/public")
    parser.add_argument("--name", default="cmu_pilot")
    parser.add_argument("--baseline", default="")
    args = parser.parse_args()
    import torch
    from ultralytics import YOLO
    data = Path(args.data).resolve()
    manifest = audit_dataset(data)
    torch.set_num_threads(args.threads)
    started = time.time()
    model = YOLO(args.weights)
    # Freeze pretrained backbone for an affordable CPU pilot; --freeze 0 on GPU.
    results = model.train(data=str(data), epochs=args.epochs, imgsz=args.imgsz,
                          batch=args.batch, device=args.device, workers=0,
                          project=str(Path(args.project).resolve()), name=args.name,
                          exist_ok=False, optimizer="AdamW", lr0=0.001, lrf=0.01,
                          freeze=args.freeze, seed=42, deterministic=True,
                          amp=args.device != "cpu", plots=True, patience=15,
                          mosaic=0.2, close_mosaic=5, degrees=25, fliplr=0.5,
                          translate=0.1, scale=0.3)
    run = Path(model.trainer.save_dir)
    best = run / "weights" / "best.pt"
    candidate = YOLO(str(best))
    validation = candidate.val(data=str(data), split="val", imgsz=args.imgsz, device=args.device,
                               workers=0, project=str(run), name="validation", plots=False)
    test = candidate.val(data=str(data), split="test", imgsz=args.imgsz, device=args.device,
                         workers=0, project=str(run), name="heldout_test", plots=True)
    baseline_metrics = None
    if args.baseline:
        baseline = YOLO(args.baseline)
        baseline_test = baseline.val(data=str(data), split="test", imgsz=args.imgsz,
                                     device=args.device, workers=0, project=str(run),
                                     name="baseline_heldout", plots=False)
        baseline_metrics = {key: float(value) for key, value in baseline_test.results_dict.items()}
    export = candidate.export(format="onnx", imgsz=args.imgsz, batch=1, half=False,
                              dynamic=False, simplify=False, opset=17, device="cpu")
    import numpy as np
    import onnxruntime as ort
    session = ort.InferenceSession(str(export), providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    output = session.run(None, {input_name: np.zeros((1, 3, args.imgsz, args.imgsz), dtype=np.float32)})[0]
    if output.shape[0] != 1 or output.shape[1] != 23 or not np.isfinite(output).all():
        raise RuntimeError(f"Unexpected web output: {output.shape}")
    report = {"created_utc": datetime.now(timezone.utc).isoformat(), "stage": "public_data_pilot",
              "checkpoint": str(best), "onnx": str(export), "onnx_sha256": hashlib.sha256(Path(export).read_bytes()).hexdigest(),
              "onnx_input": session.get_inputs()[0].shape, "onnx_output": list(output.shape),
              "dataset_source": manifest["source"], "image_counts": manifest["image_counts"],
              "foot_counts": manifest["foot_counts"], "keypoints": manifest["keypoints"],
              "actual_epochs": len(model.trainer.csv.read_text().splitlines()) - 1,
              "training_config": vars(args), "elapsed_seconds": round(time.time() - started),
              "validation": {key: float(value) for key, value in validation.results_dict.items()},
              "heldout_test": {key: float(value) for key, value in test.results_dict.items()},
              "previous_synthetic_baseline_test": baseline_metrics,
              "metric_sizing_validated": False, "production_ready": False,
              "split_policy": manifest.get("split_policy"),
              "limitations": ["No foot length/width in mm or product fit labels",
                              "K2/K3 have no ground truth; do not use them for sizing",
                              "Image split, not a verified subject-exclusive split",
                              "Training images may differ from real mobile scans",
                              "Image rights vary by source; retain dataset attribution",
                              "Real-phone tracking and metric sizing require independent validation"]}
    (run / "training_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("PUBLIC TRAINING COMPLETE", flush=True)
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile validate_public_onnx.py
"""Check exported ONNX numerical agreement on real held-out image crops."""
import argparse
import json
from pathlib import Path

import numpy as np
from PIL import Image
import torch
import onnxruntime as ort
from ultralytics import YOLO

parser = argparse.ArgumentParser()
parser.add_argument('--run', default='training/runs/public/cmu_pilot')
parser.add_argument('--images', default='training/public_cmu_pilot/images/test')
args = parser.parse_args()
run = Path(args.run)
torch.set_num_threads(4)
model = YOLO(str(run / 'weights/best.pt')).model.eval()
session = ort.InferenceSession(str(run / 'weights/best.onnx'), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
size = session.get_inputs()[0].shape[2]
records = []
for path in sorted(Path(args.images).glob('*.jpg'))[:3]:
    with Image.open(path) as image:
        image = image.convert('RGB')
        width, height = image.size
        # Preserve the whole visible image with a grey letterbox, as the current
        # web inference geometry does. Do not silently crop portrait feet away.
        scale = min(size / width, size / height)
        draw_width, draw_height = round(width * scale), round(height * scale)
        boxed = Image.new('RGB', (size, size), (114, 114, 114))
        resized = image.resize((draw_width, draw_height), Image.Resampling.BILINEAR)
        boxed.paste(resized, ((size - draw_width) // 2, (size - draw_height) // 2))
        pixels = np.asarray(boxed, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    with torch.no_grad():
        torch_output = model(torch.from_numpy(pixels))[0].numpy()
    onnx_output = session.run(None, {input_name: pixels})[0]
    if not np.isfinite(onnx_output).all() or onnx_output.shape != torch_output.shape:
        raise RuntimeError('Invalid real-image ONNX output')
    difference = np.abs(torch_output - onnx_output)
    if difference.max() > 0.01:
        raise RuntimeError(f'ONNX export disagreement: {difference.max()}')
    records.append({'image': path.name, 'shape': list(onnx_output.shape),
                    'max_absolute_difference': float(difference.max()),
                    'mean_absolute_difference': float(difference.mean())})
if len(records) != 3:
    raise RuntimeError('Three held-out images are required')
report = {'status': 'passed', 'tolerance': 0.01, 'preprocessing': 'full-image RGB letterbox, grey 114', 'records': records,
          'scope': 'Numerical export equivalence only; not mobile performance or sizing accuracy'}
(run / 'onnx_validation.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))


Default GPU run: 4,000 train, 120 validation and 120 held-out test images,
60 epochs at 256px. An image is never in more than one split. COCO does not provide
subject IDs, so subject overlap cannot be ruled out. Reduce `--train-images` for a
quick pilot. Free sessions may disconnect; download checkpoints before stopping.


In [ ]:
import subprocess, sys
from pathlib import Path
from train_public_footnet import audit_dataset
def run_stream(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

data_path = Path('/content/public_cmu/data.yaml')
if data_path.exists():
    print('Reusing audited dataset:', audit_dataset(data_path)['image_counts'])
else:
    run_stream([sys.executable, '-u', 'prepare_public_cmu.py', '--output', '/content/public_cmu',
                    '--train-images', '4000', '--val-images', '120', '--test-images', '120',
                    '--workers', '12'])


In [ ]:
run_stream([sys.executable, '-u', 'train_public_footnet.py', '--data', '/content/public_cmu/data.yaml',
                '--weights', 'yolov8n-pose.pt', '--epochs', '60', '--batch', '32',
                '--device', '0', '--freeze', '0', '--project', '/content/runs',
                '--name', 'cmu_public_gpu'])


In [ ]:
from pathlib import Path
import json, shutil
reports = sorted(Path('/content/runs').glob('*/training_report.json'), key=lambda p: p.stat().st_mtime)
if not reports:
    raise RuntimeError('Training did not finish; inspect the preceding cell.')
report_path = reports[-1]
run_stream([sys.executable, '-u', 'validate_public_onnx.py', '--run', str(report_path.parent),
            '--images', '/content/public_cmu/images/test'])
shutil.copy('/content/public_cmu/manifest.json', report_path.parent / 'dataset_manifest.json')
print(report_path.read_text())
shutil.make_archive('/content/footnet_public_results', 'zip', report_path.parent)
from google.colab import files
files.download('/content/footnet_public_results.zip')
